# 🎬 AutoShorts AI — Fábrica Automatizada de Vídeos Curtos 3D (Google Colab)

Produção ponta a ponta de curtas verticais **9:16 (1080×1920, 30 fps, H.264 + AAC)** animados 100% em código com **Three.js Toon 3D + Playwright/Chromium + FFmpeg + TTS PT-BR + Alinhamento Forçado + Legendas Dinâmicas + SFX/Música + Controle de Qualidade + Google Drive + OpenCode CLI**.

### Como usar
1. No menu **Ambiente de execução → Alterar tipo de ambiente de execução**, selecione **GPU T4** (ou superior) quando disponível.
2. (Opcional) Salve suas chaves de API em **Colab Secrets** (ícone de chave na barra lateral esquerda): `GEMINI_API_KEY`, `OPENAI_API_KEY`, `GOOGLE_API_KEY`, `OPENCODE_MODEL`.
3. Execute as células em ordem — na célula **Configuração Central**, basta informar o **TEMA** do vídeo.

> **Nota importante sobre WebGL no Colab:** ter uma GPU CUDA (ex.: T4) alocada no Colab **não garante automaticamente** que o Chromium headless esteja usando aceleração WebGL por hardware. A Célula 3 testa o renderizador WebGL real (`UNMASKED_RENDERER_WEBGL`) e avisa claramente caso esteja usando **SwiftShader (software)** em modo fallback seguro.

In [ ]:
#@title 1. Verificar Ambiente de Execução (CPU, RAM, GPU CUDA, FFmpeg, Node.js)
import os, platform, shutil, subprocess

print(f"Sistema: {platform.system()} {platform.release()} | Python: {platform.python_version()}")
print(f"CPUs lógicas: {os.cpu_count()}")

for cmd in ("nvidia-smi", "ffmpeg", "ffprobe", "node", "npm", "chromium", "google-chrome", "opencode"):
    path = shutil.which(cmd)
    print(f"  - {cmd:14s}: {path or 'NÃO ENCONTRADO'}")

if shutil.which("nvidia-smi"):
    print("\n--- GPU CUDA detectada (sistema) ---")
    print(subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], text=True))
else:
    print("\n⚠️ Nenhuma GPU NVIDIA alocada neste runtime (modo CPU).")

In [ ]:
#@title 2. Baixar Projeto e Instalar Dependências (Python + Node.js + Playwright Chromium + FFmpeg)
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/99X123/IA-geradora-de-video-roblox.git"  #@param {type:"string"}
PROJECT_DIR = Path("/content/AutoShortsAI")

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print(f"Usando diretório existente: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("Diretório ativo:", os.getcwd())

# FFmpeg + bibliotecas do Chromium no Linux/Colab
!apt-get -qq update && apt-get -qq install -y ffmpeg libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2t64 || apt-get -qq install -y ffmpeg

# Dependências Python
!pip install -q -r requirements-colab.txt

# Opcional: faster-whisper para alinhamento forçado acústico extra
INSTALAR_FASTER_WHISPER = False  #@param {type:"boolean"}
if INSTALAR_FASTER_WHISPER:
    !pip install -q faster-whisper

# Dependências Node.js (Three.js + Playwright)
!npm install --no-audit --no-fund
!npx playwright install chromium
print("\n✅ Ambiente preparado com sucesso!")

In [ ]:
#@title 3. Teste Real de GPU / WebGL no Chromium (Hardware vs SwiftShader)
import json, sys
sys.path.insert(0, os.getcwd())

from src.video_orchestrator import probe_render_environment

env_probe = probe_render_environment()
print(json.dumps(env_probe, indent=2, ensure_ascii=False))

if env_probe["gpu_webgl_confirmed"]:
    print("\n✅ WebGL acelerado por HARDWARE confirmado!")
else:
    print("\n⚠️ ATENÇÃO: Chromium está renderizando por SOFTWARE (SwiftShader/fallback).")
    print("   Dica: mantenha workers=2 e qualidade 'preview' ou 'production' conforme o tempo desejado.")

In [ ]:
#@title 4. Montar Google Drive e Criar Estrutura `Meu Drive/AutoShorts AI/`
from src.drive_manager import DriveManager

MONTAR_GOOGLE_DRIVE = True  #@param {type:"boolean"}

if MONTAR_GOOGLE_DRIVE:
    dm = DriveManager()
    drive_base = dm.mount_if_colab()
    print(f"✅ Google Drive montado e estrutura criada em: {drive_base}")
    for sub in sorted(drive_base.iterdir()):
        print("  📁", sub.name)
else:
    print("Google Drive desativado nesta execução. Os arquivos ficarão apenas no disco local do Colab.")

In [ ]:
#@title 5. Carregar Credenciais com Segurança (Colab Secrets — nunca salvas no Drive)
import os
from src.config import get_secret

SECRET_KEYS = [
    "GEMINI_API_KEY",
    "GOOGLE_API_KEY",
    "OPENAI_API_KEY",
    "OPENAI_BASE_URL",
    "GOOGLE_APPLICATION_CREDENTIALS",
    "OPENCODE_MODEL",
]

print("Status das credenciais (apenas presença, valores nunca são exibidos):")
for k in SECRET_KEYS:
    val = get_secret(k)
    if val:
        os.environ[k] = val
        print(f"  ✅ {k}: configurada")
    else:
        print(f"  ⚪ {k}: não configurada (usará alternativa/fallback quando aplicável)")

In [ ]:
#@title 6. Configuração Central da Produção (Informe apenas o tema para começar!)
TEMA = "meu robô assistente resolveu fazer café e ativou o modo turbo"  #@param {type:"string"}
DURACAO_SEGUNDOS = 60  #@param {type:"slider", min:12, max:90, step:6}
QUALIDADE = "production"  #@param ["draft", "preview", "production", "ultra"]
NUM_PERSONAGENS = 2  #@param {type:"slider", min:2, max:3, step:1}
PROVEDOR_ROTEIRO = "auto"  #@param ["auto", "gemini", "openai", "local_template"]
PROVEDOR_TTS = "auto"  #@param ["auto", "edge_tts", "google_cloud", "openai", "local_synth_test"]
ATIVAR_SFX = True  #@param {type:"boolean"}
ATIVAR_MUSICA = True  #@param {type:"boolean"}
PAUSAR_APROVACAO_STORYBOARD = False  #@param {type:"boolean"}
APROVAR_STORYBOARD = False  #@param {type:"boolean"}
WORKERS = 2  #@param {type:"slider", min:1, max:4, step:1}
NUM_PARTES = 10  #@param {type:"slider", min:2, max:16, step:1}
SALVAR_NO_DRIVE = True  #@param {type:"boolean"}
PROJECT_ID_RETOMAR = ""  #@param {type:"string"}

from src.config import AutoShortsConfig

config = AutoShortsConfig.load(
    overrides={
        "video": {
            "quality_preset": QUALIDADE,
            "target_duration_sec": DURACAO_SEGUNDOS,
            "workers": WORKERS,
            "num_parts": NUM_PARTES,
        },
        "story": {
            "num_characters": NUM_PERSONAGENS,
            "provider": PROVEDOR_ROTEIRO,
            "require_storyboard_approval": PAUSAR_APROVACAO_STORYBOARD,
        },
        "tts": {"provider": PROVEDOR_TTS},
        "audio": {"enable_sfx": ATIVAR_SFX, "enable_music": ATIVAR_MUSICA},
        "drive": {"enabled": SALVAR_NO_DRIVE and MONTAR_GOOGLE_DRIVE},
    }
)
print(f"Configuração pronta: {config.get('video.width')}x{config.get('video.height')} @ {config.get('video.fps')}fps | Tema: {TEMA}")

In [ ]:
#@title 7. Executar Fábrica Completa AutoShorts AI (Staging local rápido -> Render -> QA -> Drive)
import base64, json
from pathlib import Path
from IPython.display import HTML, Image as IPyImage, display
from src.video_orchestrator import VideoOrchestrator

# Trabalha no disco local rápido do Colab (/content/autoshorts_work) e sincroniza 1x para o Drive no final
WORK_ROOT = Path("/content/autoshorts_work")
orch = VideoOrchestrator(config=config, work_root=WORK_ROOT)

result = orch.produce(
    theme=TEMA,
    project_id=PROJECT_ID_RETOMAR.strip() or None,
    duration_sec=DURACAO_SEGUNDOS,
    approved=APROVAR_STORYBOARD,
    enable_drive_sync=SALVAR_NO_DRIVE and MONTAR_GOOGLE_DRIVE,
)

print(f"\nStatus: {result['status']} | Project ID: {result['project_id']}")

if result["status"] == "awaiting_storyboard_approval":
    print("⏸️ Pausado para aprovação do storyboard. Defina PROJECT_ID_RETOMAR =", repr(result["project_id"]), "e APROVAR_STORYBOARD = True.")
    sheet = result["storyboard_preview"]["contact_sheet"]
    display(IPyImage(filename=sheet))
else:
    qa = result["quality_report"]
    print(f"QA aprovado: {qa['approved']} | Técnicos: {qa['technical_passed']} | Falhas: {qa['failed_checks']}")
    print(f"Alinhamento: {result['alignment_method']} (fallback={result['alignment_fallback_used']})")
    if qa["visual_checks"].get("contact_sheet"):
        print("\n--- Folha de Contato (QA Visual) ---")
        display(IPyImage(filename=qa["visual_checks"]["contact_sheet"]))

    mp4_path = Path(result["video_mp4"])
    if mp4_path.exists() and mp4_path.stat().st_size < 45 * 1024 * 1024:
        b64 = base64.b64encode(mp4_path.read_bytes()).decode("ascii")
        display(HTML(f'<video controls width="360" src="data:video/mp4;base64,{b64}"></video>'))
    if result.get("drive_sync"):
        print("\n☁️ Salvo no Google Drive:", result["drive_sync"]["video_drive_path"])

In [ ]:
#@title 8. (Opcional) Re-renderizar Apenas Partes Específicas com Erro
PARTES_PARA_RERENDER = ""  #@param {type:"string"}

if PARTES_PARA_RERENDER.strip() and "project_dir" in result:
    idxs = [int(x.strip()) for x in PARTES_PARA_RERENDER.split(",") if x.strip().isdigit()]
    print(f"Re-renderizando apenas as partes: {idxs}...")
    orch.rerender_scene_parts(Path(result["project_dir"]), idxs)
    print("✅ Re-renderização parcial concluída!")
else:
    print("Nenhuma parte selecionada (ex.: informe '2,5' para re-renderizar apenas as partes 2 e 5).")

In [ ]:
#@title 9. Executar Testes Automatizados e Revisão Independente com OpenCode CLI
import json
from src.opencode_reviewer import OpenCodeReviewer

print("=== 1. Executando suíte pytest ===")
!python -m pytest -v

EXECUTAR_REVISAO_OPENCODE = False  #@param {type:"boolean"}
PERMITIR_EDICOES_OPENCODE = False  #@param {type:"boolean"}

if EXECUTAR_REVISAO_OPENCODE:
    reviewer = OpenCodeReviewer(config=config)
    oc_report = reviewer.review_project(allow_edits=PERMITIR_EDICOES_OPENCODE)
    print("\n=== Relatório OpenCode ===")
    print(json.dumps({k: v for k, v in oc_report.items() if k != "opencode_output"}, indent=2, ensure_ascii=False))
    if oc_report.get("opencode_output"):
        print("\n--- Saída do OpenCode ---")
        print(oc_report["opencode_output"][:4000])